<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c42-header.png" alt="Nextia Learning · Vision AI: See, Count and Inspect" width="100%">

# Solution: Vision-language models

**[Vision-language models](https://learning.nextia-ai.com/courses/vision/m07/vision-language-models/)** · Vision AI: See, Count and Inspect · Module 7, lesson 2 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** read the answers of two vision-language models to the same request (product type, colour, material and alt text for a catalogue photo), check whether an answer is usable structured output, and see what a small local model and a large hosted model write.

| | |
|---|---|
| **Time** | About 30 minutes |
| **Needs** | An internet connection for the setup cells. No account and no key: every model answer here was recorded once and is replayed from a file. An optional last section runs a small model on your computer (513 MB download). |
| **You should know** | JSON, and image embeddings from [Image embeddings](https://learning.nextia-ai.com/courses/vision/m03/image-embeddings/). |
| **Tested** | Python 3.12 with NumPy 2.5.4, pandas 3.1.1, Pillow 13.0.0, scikit-learn 1.9.1 and matplotlib 3.11.2 (PyTorch 2.15.0 and timm 1.0.31 where the lesson needs them), in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/vision/m07/vision-language-models/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C42/M07-L02-vision-language-models/vision-language-models-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: numpy 2.5.4, pandas 3.1.1, pillow 13.0.0, scikit-learn 1.9.1, scikit-image 0.26.0, and matplotlib for the pictures. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check numpy==2.5.4 pandas==3.1.1 pillow==13.0.0 scikit-learn==1.9.1 scikit-image==0.26.0 matplotlib==3.11.2
import warnings; warnings.filterwarnings("ignore")
import numpy; print("numpy", numpy.__version__)
import pandas; print("pandas", pandas.__version__)
import PIL; print("pillow", PIL.__version__)
import sklearn; print("scikit-learn", sklearn.__version__)
import skimage; print("scikit-image", skimage.__version__)

## Setup: get the Larkfield images

Run the next cell. It downloads the image sets that this lesson needs (about 19 MB, as zip files) into a folder `larkfield-images`, with the course's helper `compare.py`, checks each file's checksum and unpacks it. It needs an internet connection and no account, and takes about 20 seconds.

You should see `Ready:` and the number of files. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-images` (in Colab: the **Files** panel on the left), then run the cell again.

Where the photos come from: the catalogue photos are cut from Open Images photos (CC BY 2.0: the author of each photo is in `data/catalogue/manifest.csv`), the returned candles are photos of the VisA inspection set (CC BY 4.0), and the pallets are drawn by our code. Read `data/dataset-card.md` for the details.

In [ ]:
# Setup: download the image sets into larkfield-images/, check them and unpack them.
import hashlib, os, shutil, subprocess, urllib.request, zipfile
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C42/"
FILES = {
    "archives/catalogue.zip": "83752b3ce15a1b99f3b9201cc979769d7abe39633754a84865d7e60f0b3d7814",
    "archives/extras.zip": "f121137bc765866c25929a4233c557ba72b64eea0c806070cc04a86c28ec27ec",
    "compare.py": "e9794acacbce00c240906cc8b3e2256b8505ff4a8074b7507a8b159cd39b4826",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-images").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if not (path.exists() and sha256(path) == expected):
        local = HERE / name  # a copy next to the notebook, if there is one
        if local.exists():
            shutil.copy(local, path)
        else:
            download(LABS + name, path)
        if sha256(path) != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-images and run again.")
    if name.endswith(".zip"):
        with zipfile.ZipFile(path) as z:
            z.extractall(PROJECT / "data")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
n = sum(1 for p in Path("data").rglob("*") if p.is_file())
print(f"Ready: {n} files in larkfield-images/data.")

## Setup: helpers

The next cell loads NumPy as `np`, pandas as `pd`, Pillow's `Image`, matplotlib, and the course helper as `cp` (the comparison table). It defines `show(paths)`, which draws photos in a row, and the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: images, the comparison table, and checks that do not show the answer.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import compare as cp
warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)
CLASSES = ["bowl", "candle", "chair", "cushion", "lamp", "mug", "plant_pot", "vase"]

def show(paths, titles=None, size=2.2):
    """Draw photos in a row (at most 8)."""
    paths = list(paths)[:8]
    fig, axes = plt.subplots(1, len(paths), figsize=(size * len(paths), size))
    for i, (ax, p) in enumerate(zip(np.atleast_1d(axes), paths)):
        ax.imshow(Image.open(p)); ax.axis("off")
        if titles:
            ax.set_title(str(titles[i]), fontsize=8)
    plt.show()

def fingerprint(value):
    def norm(v):
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The recorded answers

The course team asked two **vision-language models** the same question about 50 catalogue test photos, 30 drawn pallets and 40 returned candles, once, and recorded every answer:

- **SmolVLM-256M-Instruct**, a small open model (Apache-2.0, 513 MB) that runs on a laptop CPU;
- a large **hosted model** behind a paid API (Azure `chat-strong`).

No photo with a person in it was sent. Each line of the files is one answer: the task, the photo, the model's raw text (`answer`) and what the recording script read from it (`parsed`). This notebook replays them: you need no key.

In [ ]:
import json
catalogue = pd.read_csv("data/catalogue/manifest.csv").set_index("image_id")
hosted = [json.loads(line) for line in open("data/recordings/vlm-hosted.jsonl")]
local = [json.loads(line) for line in open("data/recordings/vlm-local.jsonl")]
print(len(hosted), "hosted answers;", len(local), "local answers")
print(pd.Series([a["task"] for a in hosted]).value_counts().to_string())

## 2. One photo, two answers

The question for a catalogue photo asked for JSON with four keys: `product_type`, `main_colour`, `material` (each from a fixed list) and `alt_text` (one sentence for a person who cannot see the photo).

> **Predict.** Will the small model's answer be usable JSON?

In [ ]:
pid = "P-a0853ea4"
show(["data/" + catalogue.loc[pid, "file"]], [f"{pid}: labelled {catalogue.loc[pid, 'label']}"], size=3)
for name, answers in (("hosted", hosted), ("local", local)):
    a = next(x for x in answers if x["id"] == pid)
    print(f"--- {name} ({a['seconds']} s) ---")
    print(a["answer"])

The hosted answer is one line of JSON, and its alt text is right: two white cushions with a small yellow packet between them. The local answer is JSON too, but look closer: it says `bowl` and `ceramic`, and it repeats keys (`alt_text` twice, `material_text` five times). Python's `json.loads` accepts repeated keys and keeps the **last** value, so this answer's alt text becomes `"brown"`.

## 3. Is it usable structured output?

**Structured output** means that the answer has a fixed shape that code can read: here, a JSON object with three keys, each with a value from its list. Before you use an answer, check two things:

1. The whole answer is one JSON object.
2. Each key has an allowed value.

In [ ]:
TYPES = ["bowl", "candle", "chair", "cushion", "lamp", "mug", "plant_pot", "vase", "other"]
COLOURS = ["white", "black", "grey", "brown", "beige", "red", "orange", "yellow", "gold", "green", "blue",
           "purple", "pink", "clear", "silver", "multicoloured"]
MATERIALS = ["ceramic", "glass", "metal", "wood", "fabric", "wax", "plastic", "leather", "paper", "stone"]

def is_json_object(text):
    """True when the whole answer is one JSON object."""
    try:
        return isinstance(json.loads(text), dict)
    except (json.JSONDecodeError, TypeError):
        return False

print(is_json_object('{"product_type": "mug"}'), is_json_object('This is a white mug.'))

> **Your turn.** Count the local model's answers (all 120 of them) for which `is_json_object` is true. Put the number in `n_json_local`.

In [ ]:
n_json_local = sum(is_json_object(a["answer"]) for a in local)
print(n_json_local, "of", len(local))

In [ ]:
expect('n_json_local', n_json_local, 4)

> **Check.** Only 4 of the 120 local answers are one JSON object. The hosted model: 120 of 120. The small model stops after 120 new tokens, often in the middle of a list of repeated keys, so its JSON is cut off.

Now apply the second check, the allowed values, to the 50 attribute answers:

In [ ]:
def usable(text):
    """A JSON object with the three keys, each with an allowed value."""
    if not is_json_object(text):
        return False
    a = json.loads(text)
    return a.get("product_type") in TYPES and a.get("main_colour") in COLOURS and a.get("material") in MATERIALS

for name, answers in (("hosted", hosted), ("local", local)):
    attrs = [a for a in answers if a["task"] == "attributes"]
    print(f"{name:6s} usable: {sum(usable(a['answer']) for a in attrs)} of {len(attrs)}")

Usable is not the same as right: the local answer of section 2 is usable, and it calls two cushions a ceramic bowl. The next lesson measures **right** against labels.

## 4. A broken answer, read anyway

The recording script did not give up on broken answers. It took the first value of each `"key": value` pair from the text, and read a plain sentence with simple word rules. Look at one:

In [ ]:
a = next(x for x in local if x["id"] == "P-9a3fc835" and x["task"] == "attributes")
print(a["answer"][:400])
print("read as:", a["parsed"])

Reading a broken answer this way gives you *some* values, but you cannot trust that the model meant them. In a product, a broken answer should go back to the model once (a **retry**), and then to a person.

## 5. Alt text from the hosted model

**Alt text** is the short description that a screen reader reads aloud for a person who cannot see a photo.

In [ ]:
for pid in ["P-a0853ea4", "P-9e25dc07", "P-b0244e29"]:
    a = next(x for x in hosted if x["id"] == pid)
    show(["data/" + catalogue.loc[pid, "file"]], [f"labelled {catalogue.loc[pid, 'label']}"], size=2.4)
    print(a["parsed"]["alt_text"])

The third photo is labelled *lamp* in the catalogue. The model describes molten glass in a furnace. Look at the photo: which one do you believe? The next lesson comes back to this question.

## 6. Optional: run the small model yourself

This section is **off** by default. It downloads SmolVLM-256M-Instruct (513 MB) and PyTorch (about 200 MB), and runs the model on your CPU: about 5 seconds an answer on a recent laptop, more on Colab. Set `RUN_LOCAL_MODEL = True` and run the two cells. Your answers can differ from the recorded ones a little: a different CPU or package version can change a token.

Only send the model photos without people. This model runs on your computer, so the photo does not leave it. A hosted model is different: never send it a photo with a person in it.

In [ ]:
RUN_LOCAL_MODEL = False  # set to True to download and run SmolVLM-256M-Instruct (about 700 MB in all)

if RUN_LOCAL_MODEL:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch==2.15.0", "transformers==5.22.2"], check=True)
    import torch
    from transformers import AutoModelForImageTextToText, AutoProcessor
    NAME, REVISION = "HuggingFaceTB/SmolVLM-256M-Instruct", "7e3e67edbbed1bf9888184d9df282b700a323964"
    processor = AutoProcessor.from_pretrained(NAME, revision=REVISION)
    model = AutoModelForImageTextToText.from_pretrained(NAME, revision=REVISION, dtype=torch.float32).eval()
    print("loaded", NAME)
else:
    print("Skipped: RUN_LOCAL_MODEL is False.")

In [ ]:
QUESTION = ("This is a product photo from a home and garden shop. Answer with JSON only, with these keys: "
            f"\"product_type\" (one of {TYPES}), \"main_colour\" (one of {COLOURS}), \"material\" (one of "
            f"{MATERIALS}), \"alt_text\" (one sentence for a person who cannot see the photo).")

def ask(path, question=QUESTION, max_new_tokens=120):
    messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": question}]}]
    prompt = processor.apply_chat_template(messages, add_generation_prompt=True)
    x = processor(text=prompt, images=[Image.open(path).convert("RGB")], return_tensors="pt")
    with torch.no_grad():
        ids = model.generate(**x, max_new_tokens=max_new_tokens, do_sample=False)
    return processor.batch_decode(ids[:, x["input_ids"].shape[1]:], skip_special_tokens=True)[0].strip()

if RUN_LOCAL_MODEL:
    print(ask("data/" + catalogue.loc["P-a0853ea4", "file"]))
    print(ask("data/" + catalogue.loc["P-a0853ea4", "file"], "Describe this photo in one sentence."))

## Next

The [lesson page](https://learning.nextia-ai.com/courses/vision/m07/vision-language-models/) has the diagrams, the cost of each model and the knowledge checks that count toward your certificate. The next lesson measures how often each model is **right**.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Check what a vision-language model says](https://learning.nextia-ai.com/courses/vision/m07/check-what-a-vision-language-model-says/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/vision/m07/vision-language-models/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The photos are open photos (each with its author and licence in the manifests) or images made by our code; no photo shows a real person.